In [14]:
from demo.core.data import (
    SplitConfig,
    prepare_cmapss_split,
)


config = SplitConfig(
    data_path="demo/data/train_FD001.txt",

    train_ratio=0.70,

    elbow_cycle=130,
    elbow_tolerance=20,

    extended_probability=0.05,
    extended_max_extra=50,

    min_cycles_before_failure=20,

    test_fraction_min=0.10,
    test_fraction_max=0.30,

    random_seed=42,
)


data = prepare_cmapss_split(config)

In [15]:
print(f"Total engines : {len(data['engine_data'])}")
print(f"Train engines : {len(data['train_ids'])}")
print(f"Test engines  : {len(data['test_ids'])}")

for engine_id in data["train_ids"][:5]:
    print(
        engine_id,
        "→",
        len(data["train_data"][engine_id]),
        "/",
        len(data["engine_data"][engine_id])
    )


Total engines : 100
Train engines : 70
Test engines  : 30
1 → 150 / 192
2 → 118 / 287
3 → 111 / 179
4 → 119 / 189
5 → 147 / 269


In [16]:
for engine_id in data["test_ids"][:30]:
    print(
        engine_id,
        "→",
        len(data["test_data"][engine_id]),
        "/",
        len(data["engine_data"][engine_id])
    )

7 → 59 / 259
9 → 21 / 201
12 → 17 / 170
13 → 43 / 163
14 → 39 / 180
15 → 30 / 207
20 → 58 / 234
23 → 25 / 168
31 → 43 / 234
35 → 34 / 181
36 → 45 / 158
37 → 47 / 170
42 → 30 / 196
45 → 24 / 158
48 → 42 / 231
50 → 19 / 198
54 → 41 / 257
55 → 56 / 193
59 → 61 / 231
65 → 31 / 153
66 → 39 / 202
67 → 90 / 313
68 → 43 / 199
75 → 66 / 229
78 → 69 / 231
83 → 31 / 293
86 → 34 / 278
89 → 54 / 217
91 → 16 / 135
99 → 31 / 185


In [17]:
from src.preprocess_data.realtime import (
    simulate_realtime,
    RealtimeConfig,
)

realtime = simulate_realtime(
    engine_data=data["engine_data"],
    test_data=data["test_data"],
    observation_points=data["test_observation_points"],
    config=RealtimeConfig(
        update_probability=0.8,
        max_new_cycles=4,
        random_seed=42,
    ),
)

In [18]:
model, stats, normalized_data, latent_data = run_phase1(
    train_data=data["train_data"],
    n_sensors=len(data["sensors"]),
    window_len=30,
    latent_dim=16,
    bin_stride=10,
)

TypeError: run_phase1() got an unexpected keyword argument 'bin_stride'

In [ ]:
from src.phaseII.core.prepare_data import prepare_training_data
from src.phaseII.core.train import run_training

train_records, val_records = prepare_training_data(
    latent_data=latent_data,
    event_bins=event_bins,
    metadata=data["train_metadata"],
    train_ratio=0.8,
    random_seed=42,
)

model, history = run_training(
    train_records,
    val_records,
    latent_dim=16,
    hidden_dim=32,
    lam_monotonic=0.1,
    n_epochs=50,
    lr=1e-3,
    seed=42,
)

NameError: name 'latent_data' is not defined